# Azure ML Registry에서 모델 다운로드

이 노트북은 Azure ML **Registry**에 등록된 모델을 다운로드하는 방법을 설명합니다.

## 사전 요구사항
```bash
pip install azure-ai-ml azure-identity mlflow python-dotenv
```

## 환경변수 설정
`.env` 파일을 생성하고 아래 항목을 설정하세요:
```
REGISTRY_NAME=<your-registry-name>
MODEL_NAME=<your-model-name>
MODEL_VERSION=<your-model-version>
DOWNLOAD_PATH=./downloaded_model
```

In [ ]:
!pip install azure-ai-ml azure-identity mlflow python-dotenv

## 1. 설정

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

NOTEBOOK_DIR = os.path.dirname(__vsc_ipynb_file__)
load_dotenv(os.path.join(NOTEBOOK_DIR, ".env"))

REGISTRY_NAME = os.getenv("REGISTRY_NAME")  # workspace_name 대신 registry_name 사용
MODEL_NAME    = os.getenv("MODEL_NAME")
MODEL_VERSION = os.getenv("MODEL_VERSION")
DOWNLOAD_PATH = os.path.join(NOTEBOOK_DIR, os.getenv("DOWNLOAD_PATH", "./downloaded_model"))

## 2. MLClient 초기화 (Registry 연결)

> Workspace 연결과의 차이점: `workspace_name` 대신 `registry_name`만 지정하며,  
> `subscription_id`, `resource_group_name`은 **불필요**합니다.

In [3]:
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    registry_name=REGISTRY_NAME   # registry_name 으로 Registry에 연결
)

print(f"Registry: {REGISTRY_NAME}")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Registry: kichul-model-registry-ko-central


## 3. 모델 정보 확인

In [4]:
model = ml_client.models.get(name=MODEL_NAME, version=MODEL_VERSION)

print(f"모델 이름    : {model.name}")
print(f"버전         : {model.version}")
print(f"타입         : {model.type}")
print(f"Storage URI  : {model.path}")

모델 이름    : freezer-anomaly-detection-gbdt-model-py310
버전         : 11
타입         : mlflow_model
Storage URI  : https://13889717d00.blob.core.windows.net/kichul-mod-d016fb2d-2f20-5bd4-8e0f-19eab1a9249f/ExperimentRun/dcid.b3eaeff1-08b6-4c6d-b59d-80ca4bb94612/model_pipeline


## 4. 모델 다운로드

In [5]:
ml_client.models.download(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    download_path=DOWNLOAD_PATH
)

print(f"다운로드 완료: {DOWNLOAD_PATH}")

Subtype value SAS has no mapping, use base class DataReferenceCredentialDto.



다운로드 완료: ./downloaded_model


## 5. 다운로드된 파일 확인

In [ ]:
for root, dirs, files in os.walk(DOWNLOAD_PATH):
    level = root.replace(DOWNLOAD_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

downloaded_model/
  freezer-anomaly-detection-gbdt-model-py310/
    model_pipeline/
      python_env.yaml
      requirements.txt
      MLmodel
      model.pkl
      conda.yaml


## 6. MLflow로 모델 로드 (선택)

In [9]:
import mlflow

# MLmodel 파일을 찾아서 모델 경로를 동적으로 결정
model_local_path = None
for root, dirs, files in os.walk(DOWNLOAD_PATH):
    if "MLmodel" in files:
        model_local_path = root
        break

if model_local_path is None:
    raise FileNotFoundError(f"MLmodel 파일을 찾을 수 없습니다: {DOWNLOAD_PATH}")

print(f"모델 경로: {model_local_path}")
loaded_model = mlflow.sklearn.load_model(model_local_path)
print(f"모델 로드 완료: {type(loaded_model)}")

모델 경로: ./downloaded_model/freezer-anomaly-detection-gbdt-model-py310/model_pipeline
모델 로드 완료: <class 'sklearn.pipeline.Pipeline'>
